In [2]:
import sys
sys.path.append("..")  # so we can import db.py from the ml folder

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from db import engine

joins = pd.read_sql(
    """
    SELECT m."userId" AS user_id, g."stockId" AS stock_id
    FROM "Membership" m
    JOIN "Group" g ON g.id = m."groupId"
    WHERE g.type = 'COMMUNITY'
    """,
    engine,
)
stocks = pd.read_sql('SELECT id AS stock_id, symbol, sector FROM "Stock"', engine)

print(len(joins), "joins,", joins.user_id.nunique(), "users,", joins.stock_id.nunique(), "stocks")

1800 joins, 151 users, 67 stocks


In [3]:
user_ids = joins.user_id.unique()
stock_ids = joins.stock_id.unique()
u_index = {u: i for i, u in enumerate(user_ids)}
s_index = {s: i for i, s in enumerate(stock_ids)}

rows = joins.user_id.map(u_index)
cols = joins.stock_id.map(s_index)
X = csr_matrix((np.ones(len(joins)), (rows, cols)), shape=(len(user_ids), len(stock_ids)))

print("matrix shape:", X.shape)
print("filled cells:", f"{X.nnz / (X.shape[0] * X.shape[1]):.1%}")

matrix shape: (151, 67)
filled cells: 17.8%


In [4]:
sim = cosine_similarity(X.T)      # stock x stock
np.fill_diagonal(sim, 0)          # a stock is not "similar" to itself

meta = stocks.set_index("stock_id").loc[stock_ids].reset_index()  # same order as columns
sym = meta.symbol.tolist()

def similar_to(symbol, k=6):
    i = sym.index(symbol)
    top = np.argsort(sim[i])[::-1][:k]
    return pd.DataFrame({
        "symbol": [sym[j] for j in top],
        "sector": [meta.sector[j] for j in top],
        "similarity": sim[i][top].round(3),
    })

print(meta.symbol[meta.sector == "IT"].tolist()[:5])
similar_to("TCS")

['COFORGE', 'WIPRO', 'TECHM', 'INFY', 'TCS']


,symbol,sector,similarity
0,COFORGE,IT,0.513
1,INFY,IT,0.423
2,PERSISTENT,IT,0.410
3,HCLTECH,IT,0.402
4,WIPRO,IT,0.365
5,MPHASIS,IT,0.325


In [5]:
tagged = meta.sector.notna().values
hits, total = 0, 0
for i in np.where(tagged)[0]:
    top = np.argsort(sim[i])[::-1][:5]
    top = [j for j in top if tagged[j]]
    hits += sum(meta.sector[j] == meta.sector[i] for j in top)
    total += len(top)

share = meta.sector[tagged].value_counts(normalize=True)
baseline = (share ** 2).sum()  # chance of two random stocks sharing a sector

print(f"neighbours in the same sector: {hits / total:.1%}")
print(f"what pure chance would give:   {baseline:.1%}")

neighbours in the same sector: 100.0%
what pure chance would give:   12.8%


In [6]:
def recommend(user_id, k=8):
    ui = u_index[user_id]
    joined = X[ui].toarray().ravel()
    scores = joined @ sim            # sum of similarities to joined stocks
    scores[joined > 0] = -1          # never recommend what they already have
    top = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({
        "symbol": [sym[j] for j in top],
        "sector": [meta.sector[j] for j in top],
        "score": scores[top].round(2),
    })

some_user = user_ids[3]
print("joined sectors:", meta.sector[X[u_index[some_user]].toarray().ravel() > 0].value_counts().to_dict())
recommend(some_user)

joined sectors: {'Auto': 7, 'Pharma': 1}


,symbol,sector,score
0,EICHERMOT,Auto,4.24
1,TMCV,Auto,3.83
2,TATAPOWER,Energy,1.89
3,TORNTPHARM,Pharma,1.67
4,CIPLA,Pharma,1.55
5,LUPIN,Pharma,1.53
6,NMDC,Metals,1.46
7,PNB,Banking,1.34
